## Setup Tuning + Comparison

In [1]:
# --- Setup: load our toolbox ---
import numpy as np                     # numpy = calculator for lists of numbers
import pandas as pd                    # pandas = tables of data
import matplotlib.pyplot as plt        # chart drawing
import seaborn as sns                  # prettier charts
import warnings                        # hide distracting warnings
import time                            # stopwatch — we'll time how long tuning takes

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV  # + GridSearchCV = automatic settings-tester
from sklearn.compose import ColumnTransformer      # different prep per column type
from sklearn.pipeline import Pipeline              # prep + model assembly line
from sklearn.preprocessing import StandardScaler, OneHotEncoder  # scale numbers; encode categories
from sklearn.impute import SimpleImputer           # fill missing values
from sklearn.linear_model import LogisticRegression             # the L03 baseline
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier  # the two contenders
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, confusion_matrix, ConfusionMatrixDisplay  # scorecards + error tables

warnings.filterwarnings("ignore")           # keep output clean
sns.set_style("whitegrid")                  # chart style
plt.rcParams["figure.figsize"] = (11, 4.5)  # default chart size

print("✅ Libraries loaded — GridSearchCV ready")

✅ Libraries loaded — GridSearchCV ready


## Step 1 — Setup (same data + preprocessor)

In [3]:
# --- Same data, split and prep as all day — the final comparison must be apples-to-apples ---
df = pd.read_csv("data/credit_card_fraud_2026.csv")     # load the customer table
y  = df["is_fraud"]                                # the answer to predict
X  = df.drop(columns=["transaction_id", "is_fraud"])  # the clues
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42,  # identical split as L03/NB01-03
)

# Number columns vs category columns
numeric_features = [
    "amount_usd",
    "hours_since_last_txn",
    "txn_count_last_24h",
    "distance_from_home_km",
    "card_age_months",
    "customer_age",
    "account_balance_usd",
    "cvv_retry_count",
    "velocity_score",
    "time_of_day_hour",
    "day_of_week",
    "merchant_risk_score",
    "prior_disputes",
]
categorical_features = ["merchant_category", "card_type", "auth_method", "channel", "device_type", "is_foreign_transaction", "is_new_merchant", "used_vpn", "ip_country_mismatch", "billing_shipping_mismatch", "is_ai_generated_scam_attempt"]

# The usual prep recipe
preprocessor = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("scl", StandardScaler())]), numeric_features),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("enc", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]),
                      categorical_features),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)  # the fair 5-fold test used everywhere
print(f"Train: {len(X_train):,} customers · Test: {len(X_test):,} customers")

Train: 16,000 customers · Test: 4,000 customers


## Step 2 — GridSearch over Random Forest

In [4]:
# --- Tune the Random Forest: let GridSearchCV try every combination of settings for us ---
rf_pipeline = Pipeline([
    ("prep",  preprocessor),
    ("model", RandomForestClassifier(class_weight="balanced", n_jobs=-1, random_state=42)),
])

# The "menu" of settings to try — every combination gets tested
rf_grid = {
    "model__n_estimators":     [100, 200, 400],   # how many trees
    "model__min_samples_leaf": [1, 5, 20],        # how small leaves may get (bigger = less memorising)
    "model__max_depth":        [None, 10],        # tree depth cap
}
# 3 × 3 × 2 = 18 combos × 5-fold = 90 model fits

start = time.time()   # start the stopwatch
rf_search = GridSearchCV(
    rf_pipeline,
    rf_grid,
    cv=cv,          # score every combo with the same fair 5-fold test
    scoring="f1",   # judge by F1 (our balance of precision and recall)
    n_jobs=-1,      # use all CPU cores
    refit=True,     # retrain the best combo on ALL training data at the end
)
rf_search.fit(X_train, y_train)   # run all 90 fits
elapsed = time.time() - start     # stop the stopwatch

print(f"GridSearchCV finished in {elapsed:.1f}s — fit {len(rf_search.cv_results_['params'])} configurations")
print()
print(f"Best CV F1: {rf_search.best_score_:.3f}")
print(f"Best params:")
for k, v in rf_search.best_params_.items():          # loop over the winning settings
    print(f"  · {k.replace('model__', ''):20s} = {v}")

GridSearchCV finished in 20.3s — fit 18 configurations

Best CV F1: 0.324
Best params:
  · max_depth            = None
  · min_samples_leaf     = 5
  · n_estimators         = 200


## Step 3 — GridSearch over Gradient Boosting

In [5]:
# --- Tune Gradient Boosting the same way ---
gb_pipeline = Pipeline([
    ("prep",  preprocessor),
    ("model", HistGradientBoostingClassifier(class_weight="balanced", random_state=42)),
])

# GB's settings menu: step size, number of trees, tree depth
gb_grid = {
    "model__learning_rate": [0.05, 0.1, 0.2],   # size of each correction step
    "model__max_iter":      [100, 200, 400],    # number of boosting rounds (trees)
    "model__max_depth":     [4, 6, None],       # depth of each little tree
}
# 3 × 3 × 3 = 27 combos × 5-fold = 135 model fits

start = time.time()   # stopwatch on
gb_search = GridSearchCV(
    gb_pipeline,
    gb_grid,
    cv=cv,
    scoring="f1",
    n_jobs=-1,
    refit=True,     # keep the best model, retrained on all training data
)
gb_search.fit(X_train, y_train)   # run all 135 fits
elapsed = time.time() - start

print(f"GridSearchCV finished in {elapsed:.1f}s — fit {len(gb_search.cv_results_['params'])} configurations")
print()
print(f"Best CV F1: {gb_search.best_score_:.3f}")
print(f"Best params:")
for k, v in gb_search.best_params_.items():
    print(f"  · {k.replace('model__', ''):20s} = {v}")

GridSearchCV finished in 11.2s — fit 27 configurations

Best CV F1: 0.259
Best params:
  · learning_rate        = 0.2
  · max_depth            = None
  · max_iter             = 100


## Step 4 — The top-5 configurations side by side

In [6]:
# --- Look beyond the single winner: how close were the runners-up? ---
def top5(search, name):
    # Pull the full results table from the search
    df_ = pd.DataFrame(search.cv_results_)
    # Keep only the 5 best-scoring combos
    df_ = df_.sort_values("mean_test_score", ascending=False).head(5)
    rows = []
    for _, row in df_.iterrows():
        # Turn each combo's settings into one readable string
        params = ", ".join(f"{k.replace('model__','')}={v}" for k, v in row["params"].items())
        rows.append((row["mean_test_score"], row["std_test_score"], params))
    out = pd.DataFrame(rows, columns=[f"{name} CV F1", "std", "params"])
    return out

# If the top 5 scores are all within noise of each other, exact settings barely matter
print("Top 5 Random Forest configurations:")
print(top5(rf_search, "RF").to_string(index=False))
print()
print("Top 5 Gradient Boosting configurations:")
print(top5(gb_search, "GB").to_string(index=False))

Top 5 Random Forest configurations:
 RF CV F1      std                                               params
 0.323504 0.032275 max_depth=None, min_samples_leaf=5, n_estimators=200
 0.319535 0.043536 max_depth=None, min_samples_leaf=5, n_estimators=100
 0.309685 0.035293 max_depth=None, min_samples_leaf=5, n_estimators=400
 0.279786 0.032343   max_depth=10, min_samples_leaf=5, n_estimators=100
 0.279618 0.031951   max_depth=10, min_samples_leaf=5, n_estimators=200

Top 5 Gradient Boosting configurations:
 GB CV F1      std                                          params
 0.258520 0.037707 learning_rate=0.2, max_depth=None, max_iter=200
 0.258520 0.037707 learning_rate=0.2, max_depth=None, max_iter=100
 0.258520 0.037707 learning_rate=0.2, max_depth=None, max_iter=400
 0.233277 0.028145 learning_rate=0.1, max_depth=None, max_iter=200
 0.233277 0.028145 learning_rate=0.1, max_depth=None, max_iter=100


- **Neither model dominates clearly.** RF's 0.3235 best CV F1 edges out GB's 0.25852 here. The gap is small enough that the choice comes down to deployment and maintenance considerations.

## Step 5 — Train a Logistic Regression baseline for comparison

In [9]:
# --- A fair fight: give the L03 baseline the same class_weight advantage ---
lr_pipeline = Pipeline([
    ("prep",  preprocessor),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
])

cv_f1_lr = cross_val_score(lr_pipeline, X_train, y_train, cv=cv, scoring="f1", n_jobs=-1)  # fair score
lr_pipeline.fit(X_train, y_train)   # also fit on all training data for the test-set checks below

# Original NB 03 result — no class_weight, threshold=0.5 and 0.25
lr_original = Pipeline([
    ("prep",  preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=42)),
])

cv_f1_orig = cross_val_score(lr_original, X_train, y_train, cv=cv, scoring="f1", n_jobs=-1)
lr_original.fit(X_train, y_train)

# Churn probabilities from both versions of logistic regression
proba_orig     = lr_original.predict_proba(X_test)[:, 1]
proba_balanced = lr_pipeline.predict_proba(X_test)[:, 1]

print(f"NB 03 LogisticRegression (original, no class_weight)     CV F1: {cv_f1_orig.mean():.3f} ± {cv_f1_orig.std():.3f}")
print(f"NB 03 LogisticRegression (with class_weight='balanced')  CV F1: {cv_f1_lr.mean():.3f} ± {cv_f1_lr.std():.3f}")
print()

# Print a small scorecard table for each version at thresholds 0.5 and 0.25
header = f"{'Threshold':>10}  {'Precision':>10}  {'Recall':>8}  {'F1':>6}  {'Flagged':>8}"
divider = "─" * 55

for label, proba in [("Original (no class_weight)", proba_orig),
                     ("Balanced (class_weight='balanced')", proba_balanced)]:
    print(f"{label}:")
    print(header)
    print(divider)
    for t in [0.5, 0.25]:                       # try both cut-offs
        pred = (proba >= t).astype(int)         # flag customers above the cut-off
        print(f"{t:>10.2f}  {precision_score(y_test, pred, zero_division=0):>10.3f}"
              f"  {recall_score(y_test, pred):>8.3f}  {f1_score(y_test, pred):>6.3f}  {int(pred.sum()):>8}")
    print()

NB 03 LogisticRegression (original, no class_weight)     CV F1: 0.315 ± 0.068
NB 03 LogisticRegression (with class_weight='balanced')  CV F1: 0.189 ± 0.015

Original (no class_weight):
 Threshold   Precision    Recall      F1   Flagged
───────────────────────────────────────────────────────
      0.50       0.667     0.206   0.315        21
      0.25       0.569     0.485   0.524        58

Balanced (class_weight='balanced'):
 Threshold   Precision    Recall      F1   Flagged
───────────────────────────────────────────────────────
      0.50       0.107     0.809   0.188       516
      0.25       0.066     0.912   0.123       939

